In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
Data = pd.read_csv("data/train.csv")
print(Data.columns) 

# Correlational Heatmaps 

In [ ]:
plt.figure(figsize=(18,10),facecolor="black")
event_corr = Data.corr()["event"].sort_values(ascending=False).to_frame()
sns.heatmap(event_corr,annot=True)

In [ ]:
plt.figure(figsize=(21,15),facecolor="black")
plt.style.use(style="dark_background")

plt.subplot(2,2,1)
display_data1 = Data.iloc[:,0:10]
sns.heatmap(display_data1.corr(),annot=True)
plt.subplot(2,2,2)
display_data1 = Data.iloc[:,10:20]
sns.heatmap(display_data1.corr(),annot=True)
plt.subplot(2,2,3)
display_data1 = Data.iloc[:,20:30]
sns.heatmap(display_data1.corr(),annot=True)
plt.subplot(2,2,4)
display_data1 = Data.iloc[:,30:37]
sns.heatmap(display_data1.corr(),annot=True)

plt.tight_layout()

plt.figure(figsize=(18,9))
sns.heatmap(Data.corr(),annot=True,fmt=".1g")

In [ ]:
Data["time_to_hit_12"] = ((Data["event"] == 1) & (Data["time_to_hit_hours"] <= 12)).astype(int)
Data["time_to_hit_24"] = ((Data["event"] == 1) & (Data["time_to_hit_hours"] <= 24)).astype(int)
Data["time_to_hit_48"] = ((Data["event"] == 1) & (Data["time_to_hit_hours"] <= 48)).astype(int)
Data["time_to_hit_72"] = ((Data["event"] == 1) & (Data["time_to_hit_hours"] <= 72)).astype(int)


In [ ]:
print("true values for under 12hr:",(Data["time_to_hit_12"]==1).sum())
print("true values for under 24hr:",(Data["time_to_hit_24"]==1).sum())
print("true values for under 48hr:",(Data["time_to_hit_48"]==1).sum())
print("true values for under 72hr:",(Data["time_to_hit_72"]==1).sum())
print("\ntotal events that are true:",(Data["event"]==1).sum())

In [ ]:
if "event" in Data.columns:
    Data = Data.drop(["event"],axis=1)

In [ ]:
target = ["time_to_hit_12","time_to_hit_24","time_to_hit_48","time_to_hit_72"]

In [ ]:
plt.figure(figsize=(15,20),facecolor="black")
for num,feature in enumerate(target):
    target_corr = Data.corr()[feature]
    target_corr = target_corr.sort_values(ascending=False).to_frame()
    plt.subplot(2,2,num+1)
    sns.heatmap(target_corr,annot=True)
    plt.tight_layout()

In [ ]:
plt.figure(figsize=(21,13))
for num,feature in enumerate(["event_start_month","event_start_dayofweek","event_start_hour"]):
    plt.subplot(2,2,num+1)

    sns.histplot(Data[feature],discrete=True) #type:ignore

    if (feature=="event_start_month"):
        plt.xticks(labels=["jan","feb","mar","apr","may","june","jul","aug","sep","oct","nov","dec"],
                   ticks=range(12),
                   fontsize=16)

    elif (feature=="event_start_hour"):
        plt.xticks(labels=range(1,25),ticks=range(24),rotation=90) #type:ignore

    plt.xticks(fontsize = 16)   
    plt.yticks(fontsize = 16)
    plt.xlabel(feature,fontsize = 16,color = "lightblue")
    plt.ylabel("count",fontsize = 16,color = "lightblue")
    plt.grid(True,alpha=0.3)

plt.tight_layout()
    


In [ ]:
feature_selected2 = ['num_perimeters_0_5h', 'dt_first_last_0_5h',
       'low_temporal_resolution_0_5h', 'area_first_ha', 'area_growth_abs_0_5h',
       'area_growth_rel_0_5h', 'area_growth_rate_ha_per_h', 'log1p_area_first',
       'log1p_growth', 'log_area_ratio_0_5h', 'relative_growth_0_5h',
       'radial_growth_m', 'radial_growth_rate_m_per_h',
       'centroid_displacement_m', 'centroid_speed_m_per_h',
       'spread_bearing_deg', 'spread_bearing_sin', 'spread_bearing_cos',
       'dist_min_ci_0_5h', 'dist_std_ci_0_5h', 'dist_change_ci_0_5h',
       'dist_slope_ci_0_5h', 'closing_speed_m_per_h',
       'closing_speed_abs_m_per_h', 'projected_advance_m',
       'dist_accel_m_per_h2', 'dist_fit_r2_0_5h', 'alignment_cos',
       'alignment_abs', 'cross_track_component', 'along_track_speed',
       'event_start_hour', 'event_start_dayofweek', 'event_start_month']

In [ ]:
x_train = Data[feature_selected2]
y_train = Data[target]

# lasso(L1) and xgboost based importance features (gain)


In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(penalty="l1",solver="liblinear",max_iter=1000)
for num,feature in enumerate(target):
    lr.fit(X=x_train,y=Data[feature])

    rank = pd.Series(abs(lr.coef_[0]),index =x_train.columns)
    rank=(rank.sort_values(ascending=False))
    non_zero_rank=rank[rank>0]
    
    non_zero_rank.plot(kind="barh",title=f"{feature}")
    plt.show()
    print(non_zero_rank)

In [ ]:
from xgboost import XGBClassifier

x_train = Data[feature_selected2]
for num,feature in enumerate(target):
    y_train = Data[feature] 
    XGBC_fi = XGBClassifier(n_estimators = 1000,learning_rate = 0.01,n_jobs = 16)
    XGBC_fi.fit(x_train,y_train)
    booster = XGBC_fi.get_booster()
    booster = booster.get_score(importance_type="gain")
    print(f"{feature} gain is : {booster}")


In [ ]:
from sklearn.model_selection import train_test_split

x_train_12h, x_Val_12h, y_train_12h, y_Val_12h = train_test_split(
    Data[feature_selected2],
    Data[target[0]],
    test_size=0.2,        
    random_state=42,      
    stratify=Data[target[0]])

x_train_24h, x_Val_24h, y_train_24h, y_Val_24h = train_test_split(
    Data[feature_selected2],
    Data[target[1]],
    test_size=0.2,        
    random_state=42,      
    stratify=Data[target[1]])

x_train_48h, x_Val_48h, y_train_48h, y_Val_48h = train_test_split(
    Data[feature_selected2],
    Data[target[2]],
    test_size=0.2,        
    random_state=42,      
    stratify=Data[target[2]])

x_train_72h, x_Val_72h, y_train_72h, y_Val_72h = train_test_split(
    Data[feature_selected2],
    Data[target[3]],
    test_size=0.2,        
    random_state=42,      
    stratify=Data[target[3]])

# Standard/Robust scaling


In [ ]:
from sklearn.preprocessing import StandardScaler,RobustScaler
scalar = StandardScaler()
robust = RobustScaler()

def scaling(training,validation):
    x1 = robust.fit_transform(training)
    x2 = robust.transform(validation)
    return x1,x2

def Rscaling(training,validation):
    x1 = robust.fit_transform(training)
    x2 = robust.transform(validation)
    return x1,x2

In [ ]:
x_train_12h,x_Val_12h = scaling(x_train_12h,x_Val_12h)

x_train_24h,x_Val_24h = scaling(x_train_24h,x_Val_24h)

x_train_48h,x_Val_48h = Rscaling(x_train_48h,x_Val_48h)
 
x_train_72h,x_Val_72h = Rscaling(x_train_72h,x_Val_72h)

# custom kaggle metric

In [ ]:
from sklearn.metrics import make_scorer,brier_score_loss,roc_auc_score
def metric_function(y_true,y_predicted):
    if len(y_predicted.shape) == 2:
        y_probs = y_predicted[:, 1]
    else:
        y_probs = y_predicted
    c_index = roc_auc_score(y_true,y_probs)
    brier_score_inv = 1-(brier_score_loss(y_true,y_prob=y_probs)) #type:ignore

    metric = (0.3*c_index)+(0.7*brier_score_inv)

    return(metric)

custom_metric = make_scorer(metric_function,needs_proba=True)

# PERMUTATION IMPORTANCE BASED FEATURE SELECTION (12H/24H)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

rfc = RandomForestClassifier(n_estimators=150,criterion='entropy', max_depth=5, n_jobs=16)#class_weight=balanced as 24H has very low 1s in its classification.

rfc.fit(x_train_24h, y_train_24h)

result = permutation_importance(
    rfc, 
    X=x_Val_24h, 
    y=y_Val_24h, 
    n_repeats=30,   # check central limit theorm for this value 
    scoring=custom_metric
)

feature_importance_df = pd.DataFrame({
    'feature': feature_selected2,
    'importance_mean': result.importances_mean, #type:ignore
    'importance_std': result.importances_std #type:ignore
}).sort_values(by='importance_mean', ascending=True)

print(feature_importance_df)

> The features are choosen by:
* Get the values for the permutation importance mean and std(standard deviation)
* remove the scope from all the features having negative numbers (mean) and zero (mean)(here values of 0.0000__)
* remove the feature highest feature as it has the mean value of >10 times the value of the 2nd best feature
* if the std of a feature is greater than its mean remove the feature


> combination of l1 lasso and permutation importance

In [ ]:
Feature_12H = ["num_perimeters_0_5h","spread_bearing_deg","spread_bearing_sin","relative_growth_0_5h","cross_track_component","dist_min_ci_0_5h"] 

Feature_24H =["num_perimeters_0_5h","log1p_area_first","event_start_month","cross_track_component",
              "event_start_dayofweek","along_track_speed","dist_std_ci_0_5h","event_start_hour","dist_min_ci_0_5h"]

In [ ]:
plt.figure(figsize=(18,10))
sns.heatmap((Data[Feature_24H]).corr(),annot=True,cmap="bwr")

> continuation of filtering features:
* put all the features left behind in a list
* generate the heat map of the data having only the features contained in that list
* if the relation between two features is high (here >65 both +&-) from the two features remove the one which has the least importance from the list 
* repeat this process until no two features have a relation of (here >65)

# feature selection for 48H/72H (logistic regression)

In [ ]:
lr = LogisticRegression(C=0.01,solver='liblinear',max_iter=800)
lr.fit(X=x_train_48h,y=y_train_48h)


result_48_72H = permutation_importance(
    estimator=lr,
    X=x_Val_48h,
    y=y_Val_48h,
    n_jobs=16,
    n_repeats=100,
    scoring=custom_metric
)

per_imp_48_72H_result = pd.DataFrame({
    "feature": feature_selected2,
    "importance_mean" : result_48_72H.importances_mean,
    "importance_std" : result_48_72H.importances_std
}).sort_values(ascending=False,by='importance_mean')

print(per_imp_48_72H_result)

> combining the features of l1 lasso and permutation importance

In [ ]:
feature_48H =["num_perimeters_0_5h","log1p_area_first","event_start_month","cross_track_component",
              "along_track_speed","dist_std_ci_0_5h","event_start_hour","dist_min_ci_0_5h"]

feature_72H = ["event_start_month","num_perimeters_0_5h","event_start_hour","log1p_area_first",
               "cross_track_component","event_start_dayofweek","dist_min_ci_0_5h","alignment_abs"]

plt.figure(figsize=(12,5))
sns.heatmap((Data[feature_48H]).corr(),cmap='bwr',annot=True)

In [ ]:
x_train_12h, x_Val_12h, y_train_12h, y_Val_12h = train_test_split(
    Data[Feature_12H],
    Data[target[0]],
    test_size=0.2,        
    random_state=42,      
    stratify=Data[target[0]])


x_train_24h, x_Val_24h, y_train_24h, y_Val_24h = train_test_split(
    Data[Feature_24H],
    Data[target[1]],
    test_size=0.2,        
    random_state=42,      
    stratify=Data[target[1]])
x_train_24h,x_Val_24h = Rscaling(x_train_24h,x_Val_24h)

x_train_48h, x_Val_48h, y_train_48h, y_Val_48h = train_test_split(
    Data[feature_48H],
    Data[target[2]],
    test_size=0.2,        
    random_state=42,      
    stratify=Data[target[2]])

x_train_48h,x_Val_48h = Rscaling(x_train_48h,x_Val_48h)
 

x_train_72h, x_Val_72h, y_train_72h, y_Val_72h = train_test_split(
    Data[feature_72H],
    Data[target[3]],
    test_size=0.2,        
    random_state=42,      
    stratify=Data[target[3]])

x_train_72h,x_Val_72h = Rscaling(x_train_72h,x_Val_72h)


# OPTUNA OPTIMIZATION

> code for 12H/24H predictions

In [ ]:
from sklearn.metrics import log_loss
from sklearn.model_selection import RepeatedStratifiedKFold #to smoothen out the curve as the dataset is very small
from sklearn.calibration import CalibratedClassifierCV
from lightgbm import LGBMClassifier
import lightgbm as lgb
Rskf = RepeatedStratifiedKFold(n_splits=4,n_repeats=4,random_state=10)


def model(params,model,feature_set,target_set):
    score = []
    for train_id,valid_id in Rskf.split(X=feature_set,y=target_set):
        train_x,train_y = feature_set[train_id],target_set[train_id]
        valid_x,valid_y = feature_set[valid_id],target_set[valid_id]

        if(model == "xgb"):
            temp_model = XGBClassifier(**params,early_stopping_rounds =100,eval_metric = "rmse",verbose =0,tree_method = "exact")
            temp_model.fit(train_x,train_y,eval_set=[(valid_x,valid_y)],verbose = 0)
            
        
        elif(model == "lgbm"):
            temp_model = LGBMClassifier(**params,boosting_type="gbdt",objective="binary",verbose = -1)
            temp_model.fit(train_x,train_y,eval_set=[(valid_x,valid_y)],callbacks=[lgb.early_stopping(stopping_rounds=100,verbose=False),
            lgb.log_evaluation(period=False)])

        
        elif(model == "LogReg"):
            temp_model = LogisticRegression(**params,verbose=0,solver="liblinear")
            temp_model.fit(X=train_x,y=train_y)

        
        prediction = temp_model.predict_proba(valid_x) 
        metric = metric_function(valid_y,prediction)
        score.append(metric)

    return np.mean(score)

> objective function code for XGB


In [ ]:
import optuna
def objective(trial):
  params = {
    "n_estimators": trial.suggest_int("n_estimators", 300, 1500), 
    "learning_rate": trial.suggest_float("learning_rate", 0.0005, 0.01,log = True), 
    "subsample": trial.suggest_float("subsample", 0.6, 0.9), 
    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9),
    "gamma": trial.suggest_float("gamma", 1.0, 3.0), 
    "min_child_weight": trial.suggest_int("min_child_weight", 10, 50),
    "max_depth": trial.suggest_int("max_depth", 1,6),
    "reg_alpha": trial.suggest_float("reg_alpha", 0.1, 10.0, log=True),
    "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 25.0, log=True),
    "scale_pos_weight":trial.suggest_float("scale_pos_weight",1.0,5.0) #only for 24H predictions
    }
  
  score = model(params=params,model="xgb",feature_set=np.asarray(x_train_72h),target_set=np.asarray(y_train_72h))
  
  return score

> objective function code for LGBM

In [ ]:
'''def objective(trial):
    params = {
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 2, 256),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "max_depth": trial.suggest_int("max_depth", 1,5), 
        "learning_rate": trial.suggest_float("learning_rate", 0.0005, 0.01, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 5.0)
    }

    score = model(params=params,model="lgbm",feature_set=np.asarray(x_train_72h),target_set=np.asarray(y_train_72h)) 
    return score'''

> objective function code for Logistic Regression 

In [ ]:
'''def objective(trial):
    params = {
        "C" : trial.suggest_float("C",0.0001,1),
        "class_weight" : trial.suggest_categorical("class_weight",["balanced",None]),
        "max_iter" : trial.suggest_int("max_iter",100,1000),
        "penalty" : trial.suggest_categorical("penalty",["l1","l2"]),
        "tol"  : trial.suggest_float("tol",0.00001,0.1),
        "fit_intercept" : trial.suggest_categorical("fit_intercept",[True,False])
        }
    score = model(params=params,model="LogReg",feature_set=np.asarray(x_train_24h),target_set=np.asarray(y_train_24h))

    return score'''

>optuna optimization

In [ ]:
study = optuna.create_study(direction="maximize",storage="sqlite:///optuna_study_fireforest.db",study_name="forest_fire_24H_lr_V1")
study.optimize(objective,n_trials=200,n_jobs=16)#type:ignore

print(f"the best hyperparameters are:{study.best_params}")
print(f"The best custom score is (higher is better): {study.best_value}")


In [ ]:
try:
    optuna.delete_study(storage="sqlite:///optuna_study_fireforest.db",study_name="forest_fire_24H_lr_V11")
except(KeyError):
    print("the file does not exist")

> a combination of c-index and briers score (0.3*(c-index) + 0.7*(1-brier's score)). The use of brier is similar to logloss but this is not that aggressive in penalty vlaue as logloss , and here in binary classification c-index is mathametically equal to AUC-ROC score so we have used AUC-ROC score for ease.

> validation score check for 24/48/72H

In [ ]:
parameters_xgb = {'n_estimators': 973, 'learning_rate': 0.00730426284346517, 'subsample': 0.8809546171348738, 'colsample_bytree': 0.7644947108956932, 'gamma': 2.6629597317270988, 'min_child_weight': 10, 'max_depth': 1, 'reg_alpha': 0.11961843917682309, 'reg_lambda': 2.2174307140024654, 'scale_pos_weight': 3.3317982251538414}
XGB = XGBClassifier(**parameters_xgb, eval_metric="logloss",tree_method = "exact")
eval_set = [(x_train_48h, y_train_48h), (x_Val_48h, y_Val_48h)]
XGB.fit(
    x_train_48h,
    y_train_48h,
    eval_set=eval_set, 
    verbose=False)
cl = CalibratedClassifierCV(XGB,method="sigmoid",cv=3)
cl.fit(x_train_48h, y_train_48h)
xgb_pridict = cl.predict_proba(x_Val_48h)

score = metric_function(y_true=y_Val_48h,y_predicted=xgb_pridict)
print("Custom score:",score)

results = XGB.evals_result()
epochs = len(results['validation_0']['logloss'])
x_axis = range(0, epochs)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_axis, results['validation_0']['logloss'], label='Train')
ax.plot(x_axis, results['validation_1']['logloss'], label='Validation')
ax.legend()

plt.ylabel('Log Loss')
plt.title('XGBoost Learning Curve: Training vs Validation Loss')
plt.grid(True)
plt.show()

In [ ]:
lgbm_per = {'lambda_l1': 6.435376505447599e-05, 'lambda_l2': 4.3678730446096544e-07, 'num_leaves': 81, 'feature_fraction': 0.9804046802491505, 'bagging_fraction': 0.607879941675613, 'bagging_freq': 1, 'min_child_samples': 5, 'max_depth': 4, 'learning_rate': 0.009961256302623447, 'scale_pos_weight': 1.564827383668402}

lgbm_temp = LGBMClassifier(**lgbm_per,verbose =-1,boosting_type="gbdt",objective="binary")
lgbm_temp.fit(x_train_72h,y_train_72h)


lgbm_predicted_valid = lgbm_temp.predict_proba(x_Val_72h)
print("Info for 48H:")
print("valid score:",metric_function(y_true=y_Val_72h,y_predicted=lgbm_predicted_valid))

lgbm_predicted_train = lgbm_temp.predict_proba(x_train_72h)
print("train loss:",log_loss(y_true=y_train_72h,y_pred=lgbm_predicted_train))
print("validation loss:",log_loss(y_true=y_Val_72h,y_pred=lgbm_predicted_valid))
print("validation brier's_loss:",brier_score_loss(y_true=y_Val_72h,y_prob=lgbm_predicted_valid[:,1]))

>calibration check

In [ ]:
from sklearn.calibration import calibration_curve
parameters_xgb = {'n_estimators': 1482, 'learning_rate': 0.006706103763703318, 'subsample': 0.7999962614657541, 'colsample_bytree': 0.8957722383985812, 'gamma': 2.8539852629862494, 'min_child_weight': 10, 'max_depth': 5, 'reg_alpha': 0.5113758611615539, 'reg_lambda': 1.1746499967018196, 'scale_pos_weight': 4.7546811178733615}
XGB = XGBClassifier(**parameters_xgb, eval_metric="logloss",tree_method = "exact")
XGB.fit(x_train_48h,y_train_48h)
cl=CalibratedClassifierCV(XGB,method="isotonic",cv=5)
cl.fit(x_train_48h,y_train_48h)
prob_pos = cl.predict_proba(x_Val_48h)[:,1]
predicted_train = cl.predict_proba(x_train_48h)
print("valid score:",metric_function(y_true=y_Val_48h,y_predicted=prob_pos))
print("train loss:",log_loss(y_true=y_train_48h,y_pred=predicted_train))
print("validation loss:",log_loss(y_true=y_Val_48h,y_pred=prob_pos))
print("validation brier's_loss:",brier_score_loss(y_true=y_Val_48h,y_prob=prob_pos))

fraction_of_positives, mean_predicted_value = calibration_curve(y_Val_48h, prob_pos, n_bins=10)

plt.plot(mean_predicted_value, fraction_of_positives, "s-", label="xgb")
plt.plot([0, 1], [0, 1],label="Perfectly calibrated")
plt.ylabel("Fraction of positives")
plt.xlabel("Mean predicted value")
plt.title('Calibration Curve: Is the Model Honest?')
plt.legend()
plt.show()

# Testing set and results

In [ ]:
test_data = pd.read_csv("data/test.csv")
submission = pd.read_csv("data/sample_submission_Blueprint.csv")

24H:
* XGB:

the best hyperparameters are:{'n_estimators': 1383, 'learning_rate': 0.008305711121853307, 'subsample': 0.8202947188918362, 'colsample_bytree': 0.5968864781373301, 'gamma': 1.192723380191325, 'min_child_weight': 10, 'max_depth': 2, 'reg_alpha': 0.32365396052182843, 'reg_lambda': 3.1907387093730097, 'scale_pos_weight': 2.6447382870870193}

        The best custom score is (higher is better): 0.9778077554634355
        0.9635799505646057
        val_loss = 0.198

* lgbm:

the best hyperparameters are:{'lambda_l1': 0.07285729065433194, 'lambda_l2': 1.5019272427391306e-06, 'num_leaves': 171, 'feature_fraction': 0.9516737529297623, 'bagging_fraction': 0.7994327518152, 'bagging_freq': 2, 'min_child_samples': 5, 'max_depth': 2, 'learning_rate': 0.009947628654045606, 'scale_pos_weight': 4.531154420113811}

        The best custom score is (higher is better): 0.9621348472432643
        Info for 24H:
        valid score: 0.9623966657801264
        train loss: 0.19683303939671407
        validation loss: 0.22214163009581722
        validation brier's_loss: 0.05165860932509403

48H:
* XGB:

the best hyperparameters are:{'n_estimators': 1482, 'learning_rate': 0.006706103763703318, 'subsample': 0.7999962614657541, 'colsample_bytree': 0.8957722383985812, 'gamma': 2.8539852629862494, 'min_child_weight': 10, 'max_depth': 5, 'reg_alpha': 0.5113758611615539, 'reg_lambda': 1.1746499967018196, 'scale_pos_weight': 4.7546811178733615}

        The best custom score is (higher is better): 0.9822338069733205
        0.9754525808177632
        val_loss = 0.14

* lgbm:

the best hyperparameters are:{'lambda_l1': 2.888534539484371e-07, 'lambda_l2': 3.4058471874349224e-06, 'num_leaves': 129, 'feature_fraction': 0.9592100866646753, 'bagging_fraction': 0.7638259837641832, 'bagging_freq': 1, 'min_child_samples': 10, 'max_depth': 3, 'learning_rate': 0.009998371552831994, 'scale_pos_weight': 1.2478921503036036}

        The best custom score is (higher is better): 0.9753995447400691
        Info for 48H:
        valid score: 0.9681546680471278
        train loss: 0.19809393669969752
        validation loss: 0.20957437622034755
        validation brier's_loss: 0.044463111581026234

72H:
* XGB

{'n_estimators': 973, 'learning_rate': 0.00730426284346517, 'subsample': 0.8809546171348738, 'colsample_bytree': 0.7644947108956932, 'gamma': 2.6629597317270988, 'min_child_weight': 10, 'max_depth': 1, 'reg_alpha': 0.11961843917682309, 'reg_lambda': 2.2174307140024654, 'scale_pos_weight': 3.3317982251538414}
        
        The best custom score is (higher is better): 0.9890828592977042
        0.9947481069973239
        val_loss = 0.095

* lgbm:

the best hyperparameters are:{'lambda_l1': 6.435376505447599e-05, 'lambda_l2': 4.3678730446096544e-07, 'num_leaves': 81, 'feature_fraction': 0.9804046802491505, 'bagging_fraction': 0.607879941675613, 'bagging_freq': 1, 'min_child_samples': 5, 'max_depth': 4, 'learning_rate': 0.009961256302623447, 'scale_pos_weight': 1.564827383668402}

        The best custom score is (higher is better): 0.9753395589890037
        Info for 48H:
        valid score: 0.9770395947112525
        train loss: 0.18439559503381864
        validation loss: 0.18576013994716606
        validation brier's_loss: 0.032800578983924994

In [ ]:
parameters_12H_xgb= {'n_estimators': 362, 'learning_rate': 0.19605392967486865, 'subsample': 0.8734705220966648,
                    'colsample_bytree': 0.8769461141471147, 'gamma': 2.2868219191797317, 'min_child_weight': 5,
                    'max_depth': 2, 'reg_alpha': 0.10184398519668955, 'reg_lambda': 2.3842382190391684,
                    "tree_method" : "exact","eval_metric": "rmse"}


parameters_24H_xgb = {'n_estimators': 1383, 'learning_rate': 0.008305711121853307, 'subsample': 0.8202947188918362, 
                      'colsample_bytree': 0.5968864781373301, 'gamma': 1.192723380191325, 'min_child_weight': 10, 'max_depth': 2, 
                      'reg_alpha': 0.32365396052182843, 'reg_lambda': 3.1907387093730097, 'scale_pos_weight': 2.6447382870870193,
                      "tree_method" : "exact","eval_metric": "rmse"}

parameters_24H_lgbm = {'lambda_l1': 0.07285729065433194, 'lambda_l2': 1.5019272427391306e-06, 'num_leaves': 171, 
                       'feature_fraction': 0.9516737529297623, 'bagging_fraction': 0.7994327518152, 'bagging_freq': 2, 
                       'min_child_samples': 5, 'max_depth': 2, 'learning_rate': 0.009947628654045606, 'scale_pos_weight': 4.531154420113811,
                       "boosting_type":"gbdt","objective":"binary"}


parameters_48H_xgb={'n_estimators': 1482, 'learning_rate': 0.006706103763703318, 'subsample': 0.7999962614657541,
                    'colsample_bytree': 0.8957722383985812, 'gamma': 2.8539852629862494, 'min_child_weight': 10, 'max_depth': 5,
                      'reg_alpha': 0.5113758611615539, 'reg_lambda': 1.1746499967018196, 'scale_pos_weight': 4.7546811178733615,
                    "eval_metric":"rmse","tree_method" : "exact"}

parameters_48H_lgbm = {'lambda_l1': 2.888534539484371e-07, 'lambda_l2': 3.4058471874349224e-06, 'num_leaves': 129, 
                      'feature_fraction': 0.9592100866646753, 'bagging_fraction': 0.7638259837641832, 'bagging_freq': 1, 
                      'min_child_samples': 10, 'max_depth': 3, 'learning_rate': 0.009998371552831994, 'scale_pos_weight': 1.2478921503036036,
                    "boosting_type":"gbdt","objective":"binary"}


parameters_72H_xgb={'n_estimators': 973, 'learning_rate': 0.00730426284346517, 'subsample': 0.8809546171348738, 
                    'colsample_bytree': 0.7644947108956932, 'gamma': 2.6629597317270988, 'min_child_weight': 10, 
                    'max_depth': 1, 'reg_alpha': 0.11961843917682309, 'reg_lambda': 2.2174307140024654, 'scale_pos_weight': 3.3317982251538414,
                    "eval_metric":"rmse","tree_method" : "exact"}

parameters_72H_lgbm ={'lambda_l1': 6.435376505447599e-05, 'lambda_l2': 4.3678730446096544e-07, 'num_leaves': 81, 
                      'feature_fraction': 0.9804046802491505, 'bagging_fraction': 0.607879941675613, 'bagging_freq': 1,
                      'min_child_samples': 5, 'max_depth': 4, 'learning_rate': 0.009961256302623447, 'scale_pos_weight': 1.564827383668402,
                      "boosting_type":"gbdt","objective":"binary"}


In [ ]:
def xgb_model(parameters,X_train,y_train,X_test):
    Rsclaing = RobustScaler()
    scaled_train = Rsclaing.fit_transform(X=X_train)
    scaled_test = Rsclaing.transform(X=X_test)
    ensamble_model = XGBClassifier(**parameters,nthread=16)
    ensamble_model.fit(scaled_train,y_train)
    cl = CalibratedClassifierCV(ensamble_model,method="isotonic",cv=5)
    cl.fit(scaled_train,y_train)
    prediction = cl.predict_proba(scaled_test)[:,1]
    return prediction

def lgbm_model(parameters,X_train,y_train,X_test):
    Rsclaing = RobustScaler()
    scaled_train = Rsclaing.fit_transform(X=X_train)
    scaled_test = Rsclaing.transform(X=X_test)
    ensamble_model = LGBMClassifier(**parameters,n_jobs=16,verbosity=-1)
    ensamble_model.fit(scaled_train,y_train)
    cl = CalibratedClassifierCV(ensamble_model,method="sigmoid",cv=5)
    cl.fit(scaled_train,y_train)

    prediction = cl.predict_proba(scaled_test)[:,1] #type:ignore
    return prediction

In [ ]:
predictions_12_xgb = xgb_model(parameters_12H_xgb,Data[Feature_12H],Data[target[0]],test_data[Feature_12H])


predictions_24_xgb = xgb_model(parameters_24H_xgb,Data[Feature_24H],Data[target[1]],test_data[Feature_24H])
predictions_24_lgbm = lgbm_model(parameters_24H_lgbm,Data[Feature_24H],Data[target[1]],test_data[Feature_24H])


predictions_48_xgb = xgb_model(parameters_48H_xgb,Data[feature_48H],Data[target[2]],test_data[feature_48H])
predictions_48_lgbm = lgbm_model(parameters_48H_lgbm,Data[feature_48H],Data[target[2]],test_data[feature_48H])


predictions_72_xgb = xgb_model(parameters_72H_xgb,Data[feature_72H],Data[target[3]],test_data[feature_72H])
predictions_72_lgbm = lgbm_model(parameters_72H_lgbm,Data[feature_72H],Data[target[3]],test_data[feature_72H])

In [ ]:
submission["prob_12h"] = predictions_12_xgb
submission["prob_24h"] = np.clip((predictions_24_xgb),a_min=0.02,a_max=0.950)
submission["prob_48h"] = np.clip((predictions_48_xgb),a_min=0.02,a_max=0.965)
submission["prob_72h"] = np.clip((predictions_72_xgb),a_min=0.021,a_max=0.980)

In [ ]:
submission.to_csv("sample_submission.csv",index=False)